# Tesla and GameStop Stock vs Revenue Analysis

This notebook extracts historical stock price data using `yfinance` and scrapes quarterly revenue data via web scraping, then visualises both side-by-side using Plotly.

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

## Question 1: Use yfinance to Extract Tesla Stock Data

In [ ]:
# Create a Tesla Ticker object and extract full historical data
tesla = yf.Ticker('TSLA')
tesla_data = tesla.history(period='max')
tesla_data.reset_index(inplace=True)

print(f'Tesla stock data extracted: {len(tesla_data)} rows')
print(f'Date range: {tesla_data["Date"].min()} to {tesla_data["Date"].max()}')
tesla_data.head()

## Question 2: Use Webscraping to Extract Tesla Revenue Data

In [ ]:
def scrape_revenue(url, ticker_label):
    """
    Scrape quarterly revenue from Macrotrends.
    Tries BeautifulSoup first; falls back to pd.read_html if the table
    structure has changed or cannot be found.
    Returns a DataFrame with columns: Date, Revenue (numeric strings).
    """
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/114.0.0.0 Safari/537.36'
        )
    }

    response = requests.get(url, headers=headers, timeout=30)
    soup = BeautifulSoup(response.text, 'html.parser')

    revenue_df = pd.DataFrame()

    # --- Primary strategy: find the revenue table by id or class ---
    try:
        # Macrotrends embeds data in a <div> with id containing 'quarterly-revenue'
        # or inside a generic table; try multiple selectors
        tables = soup.find_all('table')
        for table in tables:
            rows = table.find_all('tr')
            data = []
            for row in rows:
                cols = row.find_all('td')
                if len(cols) >= 2:
                    date_text = cols[0].get_text(strip=True)
                    rev_text  = cols[1].get_text(strip=True)
                    data.append({'Date': date_text, 'Revenue': rev_text})
            if data:
                revenue_df = pd.DataFrame(data)
                break

        if revenue_df.empty:
            raise ValueError('No table rows parsed via BeautifulSoup')

        print(f'{ticker_label}: Revenue table found via BeautifulSoup ({len(revenue_df)} rows before cleaning).')

    except Exception as e:
        print(f'{ticker_label}: BeautifulSoup strategy failed ({e}). Falling back to pd.read_html ...')
        try:
            tables_list = pd.read_html(response.text)
            for tbl in tables_list:
                # Look for a table that has at least two columns where one
                # looks like a date and another like revenue
                tbl.columns = [str(c) for c in tbl.columns]
                if tbl.shape[1] >= 2:
                    tbl = tbl.iloc[:, :2]
                    tbl.columns = ['Date', 'Revenue']
                    # Keep only rows where Date looks like a year (starts with 20 or 19)
                    mask = tbl['Date'].astype(str).str.match(r'^(19|20)\d{2}')
                    if mask.sum() > 0:
                        revenue_df = tbl[mask].copy()
                        break
            if revenue_df.empty:
                raise ValueError('pd.read_html also found no usable table')
            print(f'{ticker_label}: Revenue table found via pd.read_html ({len(revenue_df)} rows before cleaning).')
        except Exception as e2:
            print(f'{ticker_label}: Both strategies failed: {e2}')
            return pd.DataFrame(columns=['Date', 'Revenue'])

    # --- Clean up the Revenue column ---
    revenue_df['Revenue'] = (
        revenue_df['Revenue']
        .astype(str)
        .str.replace(r'[\$,]', '', regex=True)
        .str.strip()
    )

    # Drop rows where Revenue is empty, NaN, or non-numeric
    revenue_df.replace('', pd.NA, inplace=True)
    revenue_df.dropna(subset=['Date', 'Revenue'], inplace=True)
    revenue_df = revenue_df[revenue_df['Revenue'].str.match(r'^-?\d+\.?\d*$', na=False)]
    revenue_df.reset_index(drop=True, inplace=True)

    return revenue_df


TESLA_REVENUE_URL = 'https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue'
tesla_revenue = scrape_revenue(TESLA_REVENUE_URL, 'Tesla')

print(f'\nTesla revenue data built: {len(tesla_revenue)} rows')
tesla_revenue.tail()

## Question 3: Use yfinance to Extract GameStop Stock Data

In [ ]:
# Create a GameStop Ticker object and extract full historical data
gamestop = yf.Ticker('GME')
gme_data = gamestop.history(period='max')
gme_data.reset_index(inplace=True)

print(f'GameStop stock data extracted: {len(gme_data)} rows')
print(f'Date range: {gme_data["Date"].min()} to {gme_data["Date"].max()}')
gme_data.head()

## Question 4: Use Webscraping to Extract GME Revenue Data

In [ ]:
GME_REVENUE_URL = 'https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue'
gme_revenue = scrape_revenue(GME_REVENUE_URL, 'GameStop')

print(f'\nGameStop revenue data built: {len(gme_revenue)} rows')
gme_revenue.tail()

## Graph Helper: `make_graph`

Reusable function that plots historical share price (top panel) and historical quarterly revenue (bottom panel) side-by-side, using data up through mid-2021.

In [ ]:
def make_graph(stock_data, revenue_data, stock_name):
    """
    Plot historical share price and quarterly revenue for a given stock.

    Parameters
    ----------
    stock_data   : pd.DataFrame  — yfinance history with columns Date, Close
    revenue_data : pd.DataFrame  — scraped revenue with columns Date, Revenue
    stock_name   : str           — human-readable company name (e.g. 'Tesla')
    """
    CUTOFF = '2021-06-14'  # mid-2021 cutoff

    # --- Prepare stock price data ---
    stock_df = stock_data.copy()
    stock_df['Date'] = pd.to_datetime(stock_df['Date'], utc=True).dt.tz_convert(None)
    stock_df = stock_df[stock_df['Date'] <= CUTOFF]

    # --- Prepare revenue data ---
    rev_df = revenue_data.copy()
    rev_df['Date'] = pd.to_datetime(rev_df['Date'])
    rev_df['Revenue'] = pd.to_numeric(rev_df['Revenue'], errors='coerce')
    rev_df.dropna(subset=['Revenue'], inplace=True)
    rev_df = rev_df[rev_df['Date'] <= CUTOFF]
    rev_df.sort_values('Date', inplace=True)

    # --- Build subplots ---
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=False,
        subplot_titles=(
            f'{stock_name} Historical Share Price',
            f'{stock_name} Historical Quarterly Revenue'
        ),
        vertical_spacing=0.12
    )

    # Top panel — share price line
    fig.add_trace(
        go.Scatter(
            x=stock_df['Date'],
            y=stock_df['Close'],
            mode='lines',
            name='Close Price',
            line=dict(color='royalblue', width=1.5)
        ),
        row=1, col=1
    )

    # Bottom panel — quarterly revenue bar
    fig.add_trace(
        go.Bar(
            x=rev_df['Date'],
            y=rev_df['Revenue'],
            name='Quarterly Revenue (USD M)',
            marker_color='darkorange'
        ),
        row=2, col=1
    )

    # Axis labels
    fig.update_xaxes(title_text='Date', row=1, col=1)
    fig.update_xaxes(title_text='Date', row=2, col=1)
    fig.update_yaxes(title_text='Share Price (USD)', row=1, col=1)
    fig.update_yaxes(title_text='Revenue (USD millions)', row=2, col=1)

    fig.update_layout(
        title_text=f'{stock_name} Stock Price vs Revenue Dashboard',
        title_font_size=18,
        height=800,
        showlegend=True,
        template='plotly_white'
    )

    fig.show()

print('make_graph function defined successfully.')

## Question 5: Plot Tesla Stock Graph

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6: Plot GameStop Stock Graph

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')